# Phase 3 — Exploratory Data Analysis

This notebook explores prices, ratings, discounts, review counts, and inferred product categories with tables and visualizations. Price distributions are skewed, so raw and log-transformed plots are both shown. Category labels are approximate title-based groupings.

In [1]:
# ================================================================
# HARDWAREAI - PHASE 3: EXPLORATORY DATA ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ------------------------------------------------
# 1. CONFIGURATION AND DATA LOADING
# ------------------------------------------------

DATA_PATH = "../electronics_analysis_ready.csv"
OUTPUT_DIR = "eda_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("HARDWAREAI - EXPLORATORY DATA ANALYSIS")
print("=" * 70)

print("\nDataset shape:", df.shape)
print("Total product records:", len(df))

# ------------------------------------------------
# 2. DATA OVERVIEW
# ------------------------------------------------

print("\nDATASET INFORMATION")

df.info()

print("\nNUMERICAL DESCRIPTIVE STATISTICS")

numerical_columns = [
    "discount_price",
    "actual_price",
    "review_rating",
    "no_of_ratings",
    "discount_amount",
    "discount_percent"
]

numerical_columns = [
    col for col in numerical_columns if col in df.columns
]

display(df[numerical_columns].describe().round(2))

print("\nMISSING VALUE SUMMARY")

display(
    pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        ).round(2)
    })
)

# ------------------------------------------------
# 3. PRODUCT CATEGORY CLASSIFICATION
# ------------------------------------------------

# The original category columns contain only one value.
# Therefore, we infer categories from product titles.
# These are approximate labels, not verified ground truth.

import sys
sys.path.insert(0, "..")
from backend.product_features import classify_product

df["detected_category"] = df["name"].map(classify_product)

category_counts = (
    df["detected_category"]
    .value_counts()
)

print("\nPRODUCT CATEGORY DISTRIBUTION")

display(category_counts.to_frame("Product Count"))

# Category distribution visualization

plt.figure(figsize=(12, 6))

sns.barplot(
    x=category_counts.index,
    y=category_counts.values
)

plt.title("Product Distribution by Inferred Category")
plt.xlabel("Product Category")
plt.ylabel("Number of Products")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/01_category_distribution.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 4. PRICE DISTRIBUTIONS
# ------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, column in zip(
    axes,
    ["discount_price", "actual_price"]
):

    sns.histplot(
        df[column].dropna(),
        bins=40,
        kde=True,
        ax=ax
    )

    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel("Price (INR)")
    ax.set_ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/02_price_distributions.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 5. LOG-TRANSFORMED PRICE DISTRIBUTIONS
# ------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, column in zip(
    axes,
    ["discount_price", "actual_price"]
):

    values = df[column].dropna()

    sns.histplot(
        np.log1p(values),
        bins=40,
        kde=True,
        ax=ax
    )

    ax.set_title(f"Log-Transformed {column}")
    ax.set_xlabel("log(1 + Price)")
    ax.set_ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/03_log_price_distributions.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 6. REVIEW RATING DISTRIBUTION
# ------------------------------------------------

plt.figure(figsize=(9, 5))

sns.countplot(
    data=df,
    x="review_rating"
)

plt.title("Distribution of Review Star Ratings (one sampled review per row)")
plt.xlabel("Review star rating")
plt.ylabel("Number of rows")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/04_rating_distribution.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 7. DISCOUNT PERCENTAGE DISTRIBUTION
# ------------------------------------------------

plt.figure(figsize=(10, 5))

sns.histplot(
    df["discount_percent"].dropna(),
    bins=35,
    kde=True
)

plt.title("Distribution of Discount Percentages")
plt.xlabel("Discount Percentage (%)")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/05_discount_distribution.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 8. NUMERICAL BOXPLOTS
# ------------------------------------------------

boxplot_columns = [
    "discount_price",
    "actual_price",
    "no_of_ratings",
    "discount_percent"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, column in zip(
    axes.flatten(),
    boxplot_columns
):

    sns.boxplot(
        x=df[column],
        ax=ax
    )

    ax.set_title(f"Boxplot of {column}")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/06_numerical_boxplots.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 9. CORRELATION ANALYSIS
# ------------------------------------------------

correlation_columns = [
    "discount_price",
    "actual_price",
    "review_rating",
    "no_of_ratings",
    "discount_amount",
    "discount_percent"
]

correlation_data = df[
    correlation_columns
].corr(method="pearson")

print("\nPEARSON CORRELATION MATRIX")

display(correlation_data.round(3))

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_data,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0
)

plt.title("Correlation Between Numerical Features")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/07_correlation_heatmap.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 10. PRICE VS REVIEW RATING
# ------------------------------------------------

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="discount_price",
    y="review_rating",
    alpha=0.5
)

plt.title("Discount Price vs Review Star Rating")
plt.xlabel("Discount Price (INR)")
plt.ylabel("Review star rating")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/08_price_vs_rating.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 11. REVIEW COUNT VS RATING
# ------------------------------------------------

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="no_of_ratings",
    y="review_rating",
    alpha=0.5
)

plt.title("Number of Ratings vs Review Star Rating")
plt.xlabel("Number of Ratings")
plt.ylabel("Review star rating")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/09_reviews_vs_rating.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 12. CATEGORY-WISE PRICE ANALYSIS
# ------------------------------------------------

category_price = (
    df.groupby("detected_category")["discount_price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

print("\nCATEGORY-WISE PRICE STATISTICS")

display(category_price.round(2))

plt.figure(figsize=(13, 6))

sns.barplot(
    data=df,
    x="detected_category",
    y="discount_price",
    estimator="median",
    errorbar=None
)

plt.title("Median Discount Price by Product Category")
plt.xlabel("Inferred Product Category")
plt.ylabel("Median Discount Price (INR)")
plt.xticks(rotation=40, ha="right")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/10_category_median_price.png",
    dpi=300
)

plt.show()

# ------------------------------------------------
# 13. TOP RATED PRODUCTS
# ------------------------------------------------

print("\nTOP 10 RATED PRODUCTS")

top_rated = (
    df.sort_values(
        ["review_rating", "no_of_ratings"],
        ascending=[False, False],
        na_position="last"
    )
    [["name", "discount_price",
      "review_rating", "no_of_ratings"]]
    .head(10)
)

display(top_rated)

# ------------------------------------------------
# 14. MOST REVIEWED PRODUCTS
# ------------------------------------------------

print("\nTOP 10 MOST REVIEWED PRODUCTS")

most_reviewed = (
    df.nlargest(10, "no_of_ratings")
    [["name", "discount_price",
      "review_rating", "no_of_ratings"]]
)

display(most_reviewed)

# ------------------------------------------------
# 15. AUTOMATED NUMERICAL OBSERVATIONS
# ------------------------------------------------

print("\n" + "=" * 70)
print("KEY STATISTICAL OBSERVATIONS")
print("=" * 70)

print(
    "Average discount price:",
    round(df["discount_price"].mean(), 2)
)

print(
    "Median discount price:",
    round(df["discount_price"].median(), 2)
)

print(
    "Average actual price:",
    round(df["actual_price"].mean(), 2)
)

print(
    "Median actual price:",
    round(df["actual_price"].median(), 2)
)

print(
    "Average review star rating:",
    round(df["review_rating"].mean(), 2)
)

print(
    "Median review star rating:",
    round(df["review_rating"].median(), 2)
)

print(
    "Average discount percentage:",
    round(df["discount_percent"].mean(), 2)
)

print(
    "Median discount percentage:",
    round(df["discount_percent"].median(), 2)
)

print(
    "Most frequent inferred category:",
    category_counts.idxmax()
)

print(
    "Products in most frequent category:",
    category_counts.max()
)

print(
    "Distinct inferred categories:",
    df["detected_category"].nunique()
)

# ------------------------------------------------
# 16. SAVE OUTPUTS
# ------------------------------------------------

df.to_csv(
    "../electronics_eda_dataset.csv",
    index=False
)

category_price.to_csv(
    f"{OUTPUT_DIR}/category_price_statistics.csv"
)

correlation_data.to_csv(
    f"{OUTPUT_DIR}/correlation_matrix.csv"
)

print("\n" + "=" * 70)
print("PHASE 3: EDA COMPLETED")
print("=" * 70)

print("Dataset saved: electronics_eda_dataset.csv")
print("Charts saved inside:", OUTPUT_DIR)
print("Original dataset remains unchanged.")

HARDWAREAI - EXPLORATORY DATA ANALYSIS

Dataset shape: (5010, 17)
Total product records: 5010

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5010 entries, 0 to 5009
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    5010 non-null   object 
 1   main_category           5010 non-null   object 
 2   sub_category            5010 non-null   object 
 3   image                   5010 non-null   object 
 4   link                    5010 non-null   object 
 5   no_of_ratings           4958 non-null   float64
 6   discount_price          4740 non-null   float64
 7   actual_price            4979 non-null   float64
 8   review_rating           5010 non-null   float64
 9   review_text             5007 non-null   object 
 10  discount_price_missing  5010 non-null   int64  
 11  actual_price_missing    5010 non-null   int64  
 12  review_count_missing    5010 no

,discount_price,actual_price,review_rating,no_of_ratings,discount_amount,discount_percent
count,4740.00,4979.00,5010.00,4958.00,4740.00,4740.00
mean,3078.19,5226.88,4.49,13260.09,2196.38,51.88
std,8564.87,11861.10,1.09,34803.23,4368.81,21.99
min,36.00,35.00,1.00,1.00,0.88,0.00
25%,299.00,835.00,4.00,509.25,440.75,35.04
50%,649.00,1499.00,5.00,2321.00,850.00,55.58
75%,1699.00,3999.00,5.00,9812.75,2000.00,70.02
max,127999.00,149999.00,5.00,437651.00,71875.00,97.89



MISSING VALUE SUMMARY


,Missing Count,Missing Percentage
name,0,0.00
main_category,0,0.00
sub_category,0,0.00
image,0,0.00
link,0,0.00
no_of_ratings,52,1.04
discount_price,270,5.39
actual_price,31,0.62
review_rating,0,0.00
review_text,3,0.06



PRODUCT CATEGORY DISTRIBUTION


,Product Count
detected_category,
Other / Unclassified,1076
Headphones and Earphones,592
Mobile Accessories,532
Computer Accessories,493
Chargers and Cables,486
Smartphones,345
Smartwatches,214
Batteries,183
Tablets,140


... (13 outputs left out)

,name,discount_price,review_rating,no_of_ratings
470,amazon basics high-speed hdmi cable - 10 feet ...,379.0,5.0,437651.0
605,amazon basics flexible premium hdmi cable (bla...,269.0,5.0,437651.0
1217,"amazon basics high-speed hdmi cable, 6 feet - ...",269.0,5.0,437651.0
2542,"amazon basics high-speed hdmi cable, 6 feet (2...",349.0,5.0,437651.0
170,boat bassheads 100 in ear wired earphones with...,399.0,5.0,375110.0
280,boat bassheads 100 wired in ear earphones with...,399.0,5.0,375110.0
382,boat bassheads 100 in-ear wired headphones wit...,399.0,5.0,375110.0
4365,boat bassheads 100 delhi capitals edition wire...,474.0,5.0,375110.0
4916,amazonbasics aaa performance alkaline non-rech...,639.0,5.0,351441.0
521,"redmi 9a sport (carbon black, 2gb ram, 32gb st...",6499.0,5.0,319396.0



TOP 10 MOST REVIEWED PRODUCTS


,name,discount_price,review_rating,no_of_ratings
470,amazon basics high-speed hdmi cable - 10 feet ...,379.0,5.0,437651.0
605,amazon basics flexible premium hdmi cable (bla...,269.0,5.0,437651.0
1217,"amazon basics high-speed hdmi cable, 6 feet - ...",269.0,5.0,437651.0
2542,"amazon basics high-speed hdmi cable, 6 feet (2...",349.0,5.0,437651.0
15,boat bassheads 100 in-ear wired headphones wit...,399.0,1.0,375110.0
170,boat bassheads 100 in ear wired earphones with...,399.0,5.0,375110.0
280,boat bassheads 100 wired in ear earphones with...,399.0,5.0,375110.0
382,boat bassheads 100 in-ear wired headphones wit...,399.0,5.0,375110.0
848,boat bassheads 100 wired in ear earphones with...,399.0,2.0,375110.0
4365,boat bassheads 100 delhi capitals edition wire...,474.0,5.0,375110.0



KEY STATISTICAL OBSERVATIONS
Average discount price: 3078.19
Median discount price: 649.0
Average actual price: 5226.88
Median actual price: 1499.0
Average review star rating: 4.49
Median review star rating: 5.0
Average discount percentage: 51.88
Median discount percentage: 55.58
Most frequent inferred category: Other / Unclassified
Products in most frequent category: 1076
Distinct inferred categories: 27

PHASE 3: EDA COMPLETED
Dataset saved: electronics_eda_dataset.csv
Charts saved inside: eda_outputs
Original dataset remains unchanged.


## What the plots show

- **Prices are strongly right-skewed.** The mean discount price (₹3,078) is far above the median (₹649) because a few phones, laptops and TVs cost much more than the many low-cost accessories. The log-price plots are easier to read for this reason.
- **Ratings are concentrated at 5 stars** (median 5, mean 4.49; about 74.5% of rows are 5-star). Remember that each value is one sampled review's stars, not a product average, so this plot shows review stars, not product quality.
- **Discounts are large and common**: the median discount is about 56%.
- **Correlations (Pearson):** discount price and actual price are almost perfectly related (r = 0.96), as expected. Ratings show almost no linear relationship with price (r ≈ 0.02), number of ratings (r ≈ −0.07) or discount percentage (r ≈ 0.01). Discount percentage is moderately negatively related to discount price (r ≈ −0.31): cheaper items in this sample tend to carry bigger percentage discounts.
- **Category medians** follow expectations: laptops (median ₹41,240), TVs and smartphones are the most expensive inferred groups. Categories are rule-based title labels and 1,076 rows remain unclassified.

These are associations in one historical sample. A correlation (or its absence) does not show that one variable causes another; category mix and repeated listings can also influence the numbers.